In [ ]:
1+1

2

may 27 validation

step 1

In [17]:
import sys
import os
import datetime
import pandas as pd
import importlib

# ==============================================================================
# 1. DIRECTORY SETUP
# ==============================================================================
services_path = r"C:\Users\CEGIS\Documents\GitHub\discsim\may18_validata_id_dashboard\api\services"
base_output_dir = r"C:\Users\CEGIS\Documents\GitHub\discsim\may18_validata_id_dashboard\outputs"

if services_path not in sys.path: 
    sys.path.append(services_path)

import data_generator
importlib.reload(data_generator)

preset_output_dir = os.path.join(base_output_dir, "Precalculated_Presets")
if not os.path.exists(preset_output_dir): 
    os.makedirs(preset_output_dir)

MASTER_REGISTRY_PATH = os.path.join(base_output_dir, "simulation_master_registry.csv")
def log_universe_generation(params_dict, unique_task_id, preset_name, universe_file):
    """Appends a BRAND NEW row, safely aligning column names."""
    new_row = {
        "Task_ID": unique_task_id,
        "Base_Scenario": preset_name,
        "Timestamp_Step1_Gen": datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "Universe_Data_File": universe_file,
        "Timestamp_Step2_Calc": "Pending", "n_simulations_used": "Pending", 
        "L1_Scorecard_File": "Pending", "L0_Scorecard_File": "Pending",
        "Timestamp_Step3_Eval": "Pending", "L1_Summary_File": "Pending", "L0_Summary_File": "Pending"
    }
    new_row.update(params_dict) 
    
    df_new = pd.DataFrame([new_row])
    
    # Reverting to your original logic: pd.concat forces Pandas to align columns by name
    if os.path.exists(MASTER_REGISTRY_PATH):
        df_existing = pd.read_csv(MASTER_REGISTRY_PATH)
        df_combined = pd.concat([df_existing, df_new], ignore_index=True)
        df_combined.to_csv(MASTER_REGISTRY_PATH, index=False)
    else:
        df_new.to_csv(MASTER_REGISTRY_PATH, index=False)

        
    # df_new = pd.DataFrame([new_row])
    # if os.path.exists(MASTER_REGISTRY_PATH):
    #     df_existing = pd.read_csv(MASTER_REGISTRY_PATH)
    #     df_combined = pd.concat([df_existing, df_new], ignore_index=True)
    #     df_combined.to_csv(MASTER_REGISTRY_PATH, index=False)
    # else:
    #     df_new.to_csv(MASTER_REGISTRY_PATH, index=False)

# ==============================================================================
# 2. CONFIGURATION ZONE (Easily Editable)
# ==============================================================================
base_params = {
    "n_L1s": 100, 
    "n_L0s_per_L1": 25, 
    "n_children_per_L0": 15,    
    "real_percent_stunting": 35.0, 
    "real_percent_underweight": 33.0, 
    "rho": 0.7,
    "sd_across_units_percent_under_reporting_stunting": 2.0,
    "sd_across_units_percent_under_reporting_underweight": 2.0,
    "sd_within_units_percent_under_reporting_stunting": 1.0,
    "sd_within_units_percent_under_reporting_underweight": 1.0,
    "sd_across_units_bunch_factor_haz": 0.01,
    "sd_across_units_bunch_factor_waz": 0.01,
    "sd_within_units_bunch_factor_haz": 0.01,
    "sd_within_units_bunch_factor_waz": 0.01,
    "sd_percent_copy": 2.0,
    "sd_collusion_index": 0.02,
    "mean_time_lag_L1": 15,
    "mean_time_lag_L2": 30
}

preset_scenarios = {
    "Good_L0_Good_L1": {
        "mean_percent_under_reporting_stunting": 5.0, 
        "mean_percent_under_reporting_underweight": 5.0,
        "mean_bunch_factor_haz": 0.05, 
        "mean_bunch_factor_waz": 0.05, 
        "mean_percent_copy": 5.0, 
        "mean_collusion_index": 0.05, 
        "error_sd_height_all_L0s": 0.0, 
        "error_sd_weight_all_L0s": 0.0
    },
    "Good_L0_Bad_L1": {
        "mean_percent_under_reporting_stunting": 5.0, 
        "mean_percent_under_reporting_underweight": 5.0,
        "mean_bunch_factor_haz": 0.05, 
        "mean_bunch_factor_waz": 0.05, 
        "mean_percent_copy": 70.0, 
        "mean_collusion_index": 0.20, 
        "error_sd_height_all_L0s": 0.0, 
        "error_sd_weight_all_L0s": 0.0
    },
    "Bad_L0_Good_L1": {
        "mean_percent_under_reporting_stunting": 30.0, 
        "mean_percent_under_reporting_underweight": 30.0,
        "mean_bunch_factor_haz": 0.50, 
        "mean_bunch_factor_waz": 0.50, 
        "mean_percent_copy": 5.0, 
        "mean_collusion_index": 0.05, 
        "error_sd_height_all_L0s": 0.0, 
        "error_sd_weight_all_L0s": 0.0
    },
    "Bad_L0_Bad_L1": {
        "mean_percent_under_reporting_stunting": 30.0, 
        "mean_percent_under_reporting_underweight": 30.0,
        "mean_bunch_factor_haz": 0.50, 
        "mean_bunch_factor_waz": 0.50, 
        "mean_percent_copy": 20.0, 
        "mean_collusion_index": 0.70, 
        "error_sd_height_all_L0s": 0.0, 
        "error_sd_weight_all_L0s": 0.0
    }
}

# ==============================================================================
# 3. GENERATION LOOP
# ==============================================================================
print("STEP 1: GENERATING UNIVERSES")
for preset_name, distortion_params in preset_scenarios.items():
    
    current_params = {**base_params, **distortion_params}
    
    run_timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    unique_task_id = f"{preset_name}_{run_timestamp}"
    
    try:
        child_path, df_pop = data_generator.build_universe(current_params, unique_task_id, preset_output_dir)
        log_universe_generation(current_params, unique_task_id, preset_name, child_path)
        print(f"Generated & Logged: {unique_task_id}")
    except Exception as e:
        print(f"Error on {preset_name}: {e}")

STEP 1: GENERATING UNIVERSES
   -> [Step 1] Building Synthetic Universe for task: Good_L0_Good_L1_20260607_193146
      * Simulating field fraud physics...


Assigning Personalities: 100%|██████████| 100/100 [00:00<00:00, 3527.56it/s]


Generated & Logged: Good_L0_Good_L1_20260607_193146
   -> [Step 1] Building Synthetic Universe for task: Good_L0_Bad_L1_20260607_194520
      * Simulating field fraud physics...


Assigning Personalities: 100%|██████████| 100/100 [00:00<00:00, 869.79it/s]


KeyboardInterrupt: 

In [ ]:
# step 1 vv small run skip
import sys
import os
import datetime
import pandas as pd
import importlib

# ==============================================================================
# 1. DIRECTORY SETUP
# ==============================================================================
services_path = r"C:\Users\CEGIS\Documents\GitHub\discsim\may18_validata_id_dashboard\api\services"
base_output_dir = r"C:\Users\CEGIS\Documents\GitHub\discsim\may18_validata_id_dashboard\outputs"

if services_path not in sys.path: 
    sys.path.append(services_path)

import data_generator
importlib.reload(data_generator)

preset_output_dir = os.path.join(base_output_dir, "Precalculated_Presets")
if not os.path.exists(preset_output_dir): 
    os.makedirs(preset_output_dir)

MASTER_REGISTRY_PATH = os.path.join(base_output_dir, "simulation_master_registry.csv")
def log_universe_generation(params_dict, unique_task_id, preset_name, universe_file):
    """Appends a BRAND NEW row, safely aligning column names."""
    new_row = {
        "Task_ID": unique_task_id,
        "Base_Scenario": preset_name,
        "Timestamp_Step1_Gen": datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "Universe_Data_File": universe_file,
        "Timestamp_Step2_Calc": "Pending", "n_simulations_used": "Pending", 
        "L1_Scorecard_File": "Pending", "L0_Scorecard_File": "Pending",
        "Timestamp_Step3_Eval": "Pending", "L1_Summary_File": "Pending", "L0_Summary_File": "Pending"
    }
    new_row.update(params_dict) 
    
    df_new = pd.DataFrame([new_row])
    
    # Reverting to your original logic: pd.concat forces Pandas to align columns by name
    if os.path.exists(MASTER_REGISTRY_PATH):
        df_existing = pd.read_csv(MASTER_REGISTRY_PATH)
        df_combined = pd.concat([df_existing, df_new], ignore_index=True)
        df_combined.to_csv(MASTER_REGISTRY_PATH, index=False)
    else:
        df_new.to_csv(MASTER_REGISTRY_PATH, index=False)

        
    # df_new = pd.DataFrame([new_row])
    # if os.path.exists(MASTER_REGISTRY_PATH):
    #     df_existing = pd.read_csv(MASTER_REGISTRY_PATH)
    #     df_combined = pd.concat([df_existing, df_new], ignore_index=True)
    #     df_combined.to_csv(MASTER_REGISTRY_PATH, index=False)
    # else:
    #     df_new.to_csv(MASTER_REGISTRY_PATH, index=False)

# ==============================================================================
# 2. CONFIGURATION ZONE (Easily Editable)
# ==============================================================================
base_params = {
    "n_L1s": 2, 
    "n_L0s_per_L1": 25, 
    "n_children_per_L0": 15,    
    "real_percent_stunting": 35.0, 
    "real_percent_underweight": 33.0, 
    "rho": 0.7,
    "sd_across_units_percent_under_reporting_stunting": 2.0,
    "sd_across_units_percent_under_reporting_underweight": 2.0,
    "sd_within_units_percent_under_reporting_stunting": 1.0,
    "sd_within_units_percent_under_reporting_underweight": 1.0,
    "sd_across_units_bunch_factor_haz": 0.01,
    "sd_across_units_bunch_factor_waz": 0.01,
    "sd_within_units_bunch_factor_haz": 0.01,
    "sd_within_units_bunch_factor_waz": 0.01,
    "sd_percent_copy": 2.0,
    "sd_collusion_index": 0.02,
    "mean_time_lag_L1": 15,
    "mean_time_lag_L2": 30
}

preset_scenarios = {
    "Good_L0_Good_L1": {
        "mean_percent_under_reporting_stunting": 5.0, 
        "mean_percent_under_reporting_underweight": 5.0,
        "mean_bunch_factor_haz": 0.05, 
        "mean_bunch_factor_waz": 0.05, 
        "mean_percent_copy": 5.0, 
        "mean_collusion_index": 0.05, 
        "error_sd_height_all_L0s": 0.0, 
        "error_sd_weight_all_L0s": 0.0
    },
    "Good_L0_Bad_L1": {
        "mean_percent_under_reporting_stunting": 5.0, 
        "mean_percent_under_reporting_underweight": 5.0,
        "mean_bunch_factor_haz": 0.05, 
        "mean_bunch_factor_waz": 0.05, 
        "mean_percent_copy": 70.0, 
        "mean_collusion_index": 0.20, 
        "error_sd_height_all_L0s": 0.0, 
        "error_sd_weight_all_L0s": 0.0
    },
    "Bad_L0_Good_L1": {
        "mean_percent_under_reporting_stunting": 30.0, 
        "mean_percent_under_reporting_underweight": 30.0,
        "mean_bunch_factor_haz": 0.50, 
        "mean_bunch_factor_waz": 0.50, 
        "mean_percent_copy": 5.0, 
        "mean_collusion_index": 0.05, 
        "error_sd_height_all_L0s": 0.0, 
        "error_sd_weight_all_L0s": 0.0
    },
    "Bad_L0_Bad_L1": {
        "mean_percent_under_reporting_stunting": 30.0, 
        "mean_percent_under_reporting_underweight": 30.0,
        "mean_bunch_factor_haz": 0.50, 
        "mean_bunch_factor_waz": 0.50, 
        "mean_percent_copy": 20.0, 
        "mean_collusion_index": 0.70, 
        "error_sd_height_all_L0s": 0.0, 
        "error_sd_weight_all_L0s": 0.0
    }
}

# ==============================================================================
# 3. GENERATION LOOP
# ==============================================================================
print("STEP 1: GENERATING UNIVERSES")
for preset_name, distortion_params in preset_scenarios.items():
    
    current_params = {**base_params, **distortion_params}
    
    run_timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    unique_task_id = f"{preset_name}_{run_timestamp}"
    
    try:
        child_path, df_pop = data_generator.build_universe(current_params, unique_task_id, preset_output_dir)
        log_universe_generation(current_params, unique_task_id, preset_name, child_path)
        print(f"Generated & Logged: {unique_task_id}")
    except Exception as e:
        print(f"Error on {preset_name}: {e}")

STEP 1: GENERATING UNIVERSES
   -> [Step 1] Building Synthetic Universe for task: Good_L0_Good_L1_20260607_191935
      * Booting parallel engine using 10 of 20 available CPU cores...
Generated & Logged: Good_L0_Good_L1_20260607_191935
   -> [Step 1] Building Synthetic Universe for task: Good_L0_Bad_L1_20260607_191943
      * Booting parallel engine using 10 of 20 available CPU cores...
Generated & Logged: Good_L0_Bad_L1_20260607_191943
   -> [Step 1] Building Synthetic Universe for task: Bad_L0_Good_L1_20260607_191950
      * Booting parallel engine using 10 of 20 available CPU cores...
Generated & Logged: Bad_L0_Good_L1_20260607_191950
   -> [Step 1] Building Synthetic Universe for task: Bad_L0_Bad_L1_20260607_191958
      * Booting parallel engine using 10 of 20 available CPU cores...
Generated & Logged: Bad_L0_Bad_L1_20260607_191958


step 2

In [18]:
# step 2 v3 code
import sys
import os
import datetime
import pandas as pd
import importlib

# ==============================================================================
# CONFIGURATION
# ==============================================================================
services_path = r"C:\Users\CEGIS\Documents\GitHub\discsim\may18_validata_id_dashboard\api\services"
base_output_dir = r"C:\Users\CEGIS\Documents\GitHub\discsim\may18_validata_id_dashboard\outputs"
if services_path not in sys.path: sys.path.append(services_path)

import metrics_calculator
importlib.reload(metrics_calculator)

preset_output_dir = os.path.join(base_output_dir, "Precalculated_Presets")
MASTER_REGISTRY_PATH = os.path.join(base_output_dir, "simulation_master_registry.csv")

# Identify the parent Universe runs you want to use as baselines
target_tasks = [
    "Bad_L0_Bad_L1_20260604_121926"
    # "Bad_L0_Bad_L1_20260606_134031"
        # "Good_L0_Good_L1_20260601_205913",
        # "Good_L0_Bad_L1_20260601_210701",
        # "Bad_L0_Good_L1_20260601_212455",
        # "Bad_L0_Bad_L1_20260601_213328",
    # "Bad_L0_Bad_L1_20260601_231329"
    # "Good_L0_Good_L1_20260601_205913",
    # "Good_L0_Bad_L1_20260601_210701",
    # "Bad_L0_Good_L1_20260601_212455",
    # "Bad_L0_Bad_L1_20260601_213328"
]

N_SIMULATIONS = 1 # Number of Monte Carlo loops

# ==============================================================================
# EXECUTION
# ==============================================================================
print("STEP 2: RUNNING NEW OPERATIONAL MATRIX ITERATIONS")

try:
    registry_df = pd.read_csv(MASTER_REGISTRY_PATH)
except FileNotFoundError:
    print(f"Registry not found at {MASTER_REGISTRY_PATH}")
    sys.exit()

# FIX: Loop through each specific task in the list
for PARENT_UNIVERSE_RUN in target_tasks:
    print(f"\n{'-'*60}")
    print(f"Processing Parent Universe: {PARENT_UNIVERSE_RUN}")
    print(f"{'-'*60}")
    
    # Now we compare string to string, which Pandas handles perfectly
    parent_row = registry_df[registry_df['Task_ID'] == PARENT_UNIVERSE_RUN]
    
    if parent_row.empty:
        print(f"❌ Parent Task_ID '{PARENT_UNIVERSE_RUN}' not found in registry. Skipping.")
        continue

    # Get the file path. Note: in your Step 1 log it was called 'Universe_Data_File'
    universe_file_path = parent_row.iloc[0].get('Universe_Data_File')

    if not os.path.exists(str(universe_file_path)):
        print(f"❌ Population file does not exist at: {universe_file_path}. Skipping.")
        continue

    # Load the parent universe data
    print("   * Loading Universe data into memory...")
    df_universe = pd.read_parquet(str(universe_file_path))

    # GENERATE A FRESH TIMESTAMP FOR THE NEW ROW
    current_time = datetime.datetime.now()
    timestamp_str = current_time.strftime("%Y%m%d_%H%M%S")
    
    # Keep the base scenario name but assign a unique runtime signature
    base_scenario = PARENT_UNIVERSE_RUN.split('_202')[0]
    new_task_id = f"{base_scenario}_{timestamp_str}_Step2_Iter"

    print(f"   * Spawning new unique run: {new_task_id}")

    try:
        # Run Step 2 using the new task id token
        l0_matrix_path, l1_matrix_path = metrics_calculator.run_tracer_engine(
            df_pop=df_universe, 
            task_id=new_task_id, 
            output_dir=preset_output_dir,
            n_simulations=N_SIMULATIONS,
            indicators=["Height", "Weight"]
        )
        
        # CONSTRUCT THE NEW REGISTRY ROW
        new_row = {
            'Task_ID': new_task_id,
            'Base_Scenario': base_scenario,
            'Universe_Data_File': str(universe_file_path),
            'L0_Scorecard_File': str(l0_matrix_path),
            'L1_Scorecard_File': str(l1_matrix_path),
            'L0_Summary_File': 'Pending',
            'L1_Summary_File': 'Pending',
            'Timestamp_Step1_Gen': parent_row.iloc[0].get('Timestamp_Step1_Gen'),
            'Timestamp_Step2_Calc': current_time.strftime("%Y-%m-%d %H:%M:%S"),
            'Timestamp_Step3_Eval': 'Pending',
            'n_simulations_used': N_SIMULATIONS
        }
        
        # Pull over any parameters from the parent row so the new row has all the config data
        for col in registry_df.columns:
            if col not in new_row and col in parent_row.columns:
                new_row[col] = parent_row.iloc[0][col]
        
        # Append the row directly to the master registry dataframe
        registry_df = pd.concat([registry_df, pd.DataFrame([new_row])], ignore_index=True)
        registry_df.to_csv(MASTER_REGISTRY_PATH, index=False)
        
        print(f"✅ SUCCESS! New row added to registry for Task_ID: {new_task_id}")
        
    except Exception as e:
        print(f"❌ Error during Step 2 iteration for {PARENT_UNIVERSE_RUN}: {e}")

print("\n🎉 ALL STEP 2 TASKS COMPLETE.")

STEP 2: RUNNING NEW OPERATIONAL MATRIX ITERATIONS

------------------------------------------------------------
Processing Parent Universe: Bad_L0_Bad_L1_20260604_121926
------------------------------------------------------------
   * Loading Universe data into memory...
   * Spawning new unique run: Bad_L0_Bad_L1_20260608_213154_Step2_Iter
   -> [Step 2] Executing 3D Tensor Math Engine for Task: Bad_L0_Bad_L1_20260608_213154_Step2_Iter
      * Processing Indicator: Height (Tensor Accelerated)...
      * Processing Indicator: Weight (Tensor Accelerated)...
   -> [Step 2] Complete. Matrix Files Extracted to:
      1. C:\Users\CEGIS\Documents\GitHub\discsim\may18_validata_id_dashboard\outputs\Precalculated_Presets\calculated_metrics_L0_Bad_L0_Bad_L1_20260608_213154_Step2_Iter.parquet
      2. C:\Users\CEGIS\Documents\GitHub\discsim\may18_validata_id_dashboard\outputs\Precalculated_Presets\calculated_metrics_L1_Bad_L0_Bad_L1_20260608_213154_Step2_Iter.parquet
✅ SUCCESS! New row added to 

step 3 

In [19]:
# step 3 v2

import sys
import os
import datetime
import pandas as pd
import importlib

# ==============================================================================
# 1. DIRECTORY SETUP
# ==============================================================================
services_path = r"C:\Users\CEGIS\Documents\GitHub\discsim\may18_validata_id_dashboard\api\services"
base_output_dir = r"C:\Users\CEGIS\Documents\GitHub\discsim\may18_validata_id_dashboard\outputs"
if services_path not in sys.path: sys.path.append(services_path)

import analytics_ranker
importlib.reload(analytics_ranker)

preset_output_dir = os.path.join(base_output_dir, "Precalculated_Presets")
MASTER_REGISTRY_PATH = os.path.join(base_output_dir, "simulation_master_registry.csv")

# ==============================================================================
# 2. MANUAL TARGETS
# ==============================================================================
# NOTE: Make sure these match the NEW timestamped Task_IDs generated by your Step 2
TARGET_CALCULATED_RUNS = [
    "Bad_L0_Bad_L1_20260608_213154_Step2_Iter"
    # "Bad_L0_Bad_L1_20260606_225327_Step2_Iter",
    # "Bad_L0_Bad_L1_20260606_224555_Step2_Iter",
    # "Bad_L0_Bad_L1_20260606_223622_Step2_Iter"
    # "Bad_L0_Bad_L1_20260606_221222_Step2_Iter"
    # "Bad_L0_Bad_L1_20260606_134118_Step2_Iter"
    # "Good_L0_Bad_L1_20260603_221306_Step2_Iter",
    # "Bad_L0_Good_L1_20260604_001321_Step2_Iter",
    # "Bad_L0_Bad_L1_20260604_021251_Step2_Iter"
# "Good_L0_Good_L1_20260602_003532_Step2_Iter",
# "Good_L0_Bad_L1_20260602_010620_Step2_Iter",
# "Bad_L0_Good_L1_20260602_013747_Step2_Iter",
# "Bad_L0_Bad_L1_20260602_020907_Step2_Iter"
#     "Good_L0_Good_L1_20260601_205913",
# "Good_L0_Bad_L1_20260601_210701",
# "Bad_L0_Good_L1_20260601_212455",
# "Bad_L0_Bad_L1_20260601_213328",

    # "Bad_L0_Bad_L1_20260601_231408_Step2_Iter"
    # # "Good_L0_Good_L1_20260528_145250_Calc_1sims"#,
    #  "Good_L0_Bad_L1_20260528_150009_Calc_1sims",
    #  "Bad_L0_Good_L1_20260528_151136_Calc_1sims",
    #  "Bad_L0_Bad_L1_20260528_152028_Calc_1sims"
]

# ==============================================================================
# 3. EVALUATION ENGINE (APPENDS NEW ROWS)
# ==============================================================================
print("STEP 3: EVALUATING FINAL RESULTS (MANUAL TARGET MODE)")
try:
    registry_df = pd.read_csv(MASTER_REGISTRY_PATH)
except FileNotFoundError:
    print(f"Cannot find Registry at {MASTER_REGISTRY_PATH}. Run Step 1 first!")
    sys.exit()

target_runs = registry_df[registry_df['Task_ID'].isin(TARGET_CALCULATED_RUNS)]

if target_runs.empty:
    print("No matching Task_IDs found in the registry. Did you copy the names correctly?")
    sys.exit()

for index, row in target_runs.iterrows():
    task_id = row['Task_ID']
    l1_scorecard_path = row.get('L1_Scorecard_File')
    l0_scorecard_path = row.get('L0_Scorecard_File')
    
    if pd.isna(l1_scorecard_path) or str(l1_scorecard_path).strip() == "Pending":
        print(f"Skipping {task_id}: Step 2 (Scorecards) not calculated yet.")
        continue
        
    # FIX 1: THE SKIP CHECK FOR "ALREADY EXISTS" HAS BEEN COMPLETELY REMOVED

    print(f"\n=======================================================")
    print(f"Evaluating Overlap Accuracies for: {task_id}...")
    print(f"=======================================================")
    
    if not os.path.exists(str(l1_scorecard_path)) or not os.path.exists(str(l0_scorecard_path)):
        print(f"Could not find Scorecard matrices for {task_id}. Skipping...")
        continue
        
    try:
        # FIX 2: CREATE A NEW TIMESTAMPED TASK_ID FOR STEP 3
        eval_timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
        eval_task_id = f"{task_id}_Eval_{eval_timestamp}"
        
        path_l0_summary, path_l1_summary = analytics_ranker.process_ranking_analytics(
            l0_parquet_path=str(l0_scorecard_path), 
            l1_parquet_path=str(l1_scorecard_path), 
            output_dir=preset_output_dir, 
            task_id=eval_task_id, # Pass the NEW timestamped ID to the ranker
            target_percentiles=[0.10, 0.20, 0.30,0.40,0.50,0.60,0.70,0.80,0.90]
        )
        
        # FIX 3: CREATE A NEW ROW AND APPEND IT INSTEAD OF OVERWRITING
        new_row = row.copy()
        new_row['Task_ID'] = eval_task_id
        new_row['L0_Summary_File'] = str(path_l0_summary)
        new_row['L1_Summary_File'] = str(path_l1_summary)
        new_row['Timestamp_Step3_Eval'] = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        
        registry_df = pd.concat([registry_df, pd.DataFrame([new_row])], ignore_index=True)
        registry_df.to_csv(MASTER_REGISTRY_PATH, index=False)
        print(f"Evaluation Complete & New Row Appended for {eval_task_id}")
        
    except Exception as e:
        print(f"Error evaluating {task_id}: {e}")

print("\nSTEP 3 COMPLETE! The Master Registry has been updated with new rows.")

STEP 3: EVALUATING FINAL RESULTS (MANUAL TARGET MODE)

Evaluating Overlap Accuracies for: Bad_L0_Bad_L1_20260608_213154_Step2_Iter...
   -> [Step 3] Executing Analytics Ranking Engine for Task: Bad_L0_Bad_L1_20260608_213154_Step2_Iter_Eval_20260608_213214
      * Reading precalculated matrices from Step 2...
      * Evaluating Clinic (L0) catching accuracies (V2) [Vectorized]...
      * Evaluating Supervisor (L1) catching accuracies (V1, V3) [Vectorized]...
   -> [Step 3] Complete. Unified Final DB Exported to:
      - C:\Users\CEGIS\Documents\GitHub\discsim\may18_validata_id_dashboard\outputs\Precalculated_Presets\Tracer_Master_DB_Bad_L0_Bad_L1_20260608_213154_Step2_Iter_Eval_20260608_213214.parquet
      - C:\Users\CEGIS\Documents\GitHub\discsim\may18_validata_id_dashboard\outputs\Precalculated_Presets\Tracer_Master_DB_Bad_L0_Bad_L1_20260608_213154_Step2_Iter_Eval_20260608_213214.csv
Evaluation Complete & New Row Appended for Bad_L0_Bad_L1_20260608_213154_Step2_Iter_Eval_20260608_213